In [92]:
import nltk
import re
import json
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline, Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')
from tqdm import tqdm

import lib

log = lib.getLogger('yelp_sentiments')
EXPORT_PATH = Path().cwd() / 'sentiments'

# TODO: Explore and find a way to pre-process and generate a word cloud and sentiment analysis with vader and roberta

[nltk_data] Error loading vader_lexicon: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1035)>


In [19]:
df_review_text = lib.read_data('/Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/yelp/yelp_dataset/clean/yelp_review_CLEAN.csv')
# df_review = lib.read_data('/Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/yelp/yelp_dataset/clean/yelp_review_CLEAN.csv')

PortfolioLogger.lib.tools: INFO: Encoding: utf-8
PortfolioLogger.lib.tools: INFO: Stripping whitespaces from yelp_review_CLEAN.csv
PortfolioLogger.lib.performance: INFO: <function read_data at 0x10fe8b6a0> took 26.151 mins to complete.


In [20]:
df_review_text.columns

Index(['Unnamed: 0', 'review_id', 'user_id', 'business_id', 'stars', 'useful',
       'funny', 'cool', 'text', 'date'],
      dtype='object')

In [22]:
grouped = df_review_text.groupby('business_id')

In [88]:
grouped.size()

business_id
---kPU91CF4Lq2-WlRu9Lw    24
--0iUa4sNDFiZFrAdIWhZQ    14
--30_8IhuyMHbSOcNWd6DQ     9
--7PUidqRWpRSpXebiyxTg    12
--7jw19RH9JKXgFohspgQw    13
                          ..
zznZqH9CiAznbkV6fXyHWA    12
zztOG2cKm87I6Iw_tleZsQ     6
zzu6_r3DxBJuXcjnOYVdTw     8
zzw66H6hVjXQEt0Js3Mo4A     5
zzyx5x0Z7xXWWvWnZFuxlQ     8
Length: 150346, dtype: int64

# Vader Sentiment

In [ ]:
def get_sentiment(id, df):
    model_name = "cardiffnlp/twitter-roberta-base-sentiment"
    tokenizer = AutoTokenizer.from_pretrained(model_name, truncation=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)

    roberta_base_remap = {
        'LABEL_2': 'Positive',
        'LABEL_1': 'Neutral',
        'LABEL_0': 'Negative'
    }

    # Get a bag of words
    vectorizer = CountVectorizer()
    bag_of_words = vectorizer.fit_transform(df['text'].tolist())
    sentiment_words = vectorizer.get_feature_names_out().tolist().join(', ')

    # cleaned = re.sub(r"[^A-Za-z\s]", " ", all_text)  # keep only letters + spaces
    # cleaned = re.sub(r"\s+", " ", cleaned).strip()   # collapse spaces

    sentiment = pipeline(
        "sentiment-analysis",
        model=model,
        tokenizer=tokenizer,
        truncation=True,
        batch_size=32,
        padding="max_length",   # <—— makes everything exactly 512
        max_length=512,         # <—— hard limit
        device=0  # GPU = 0, CPU = -1
    )

    out = sentiment(sentiment_words)
    print('asdasdf', out)

    result = {
        'sentiment': roberta_base_remap[out['label']],
        'score': out['score'],
        'text': sentiment_words
    }

    with open(EXPORT_PATH / f'{id}.json', 'a') as f:
        json.dump(result, f)

# Roberta Base Sentiment

In [89]:
def get_sentiment(id, df):
    model_name = "cardiffnlp/twitter-roberta-base-sentiment"
    tokenizer = AutoTokenizer.from_pretrained(model_name, truncation=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)

    roberta_base_remap = {
        'LABEL_2': 'Positive',
        'LABEL_1': 'Neutral',
        'LABEL_0': 'Negative'
    }

    # Get a bag of words
    vectorizer = CountVectorizer()
    bag_of_words = vectorizer.fit_transform(df['text'].tolist())
    sentiment_words = vectorizer.get_feature_names_out().tolist().join(', ')

    # cleaned = re.sub(r"[^A-Za-z\s]", " ", all_text)  # keep only letters + spaces
    # cleaned = re.sub(r"\s+", " ", cleaned).strip()   # collapse spaces

    sentiment = pipeline(
        "sentiment-analysis",
        model=model,
        tokenizer=tokenizer,
        truncation=True,
        batch_size=32,
        padding="max_length",   # <—— makes everything exactly 512
        max_length=512,         # <—— hard limit
        device=0  # GPU = 0, CPU = -1
    )

    out = sentiment(sentiment_words)
    print('asdasdf', out)

    result = {
        'sentiment': roberta_base_remap[out['label']],
        'score': out['score'],
        'text': sentiment_words
    }

    with open(EXPORT_PATH / f'{id}.json', 'a') as f:
        json.dump(result, f)

In [90]:
for id, df in tqdm(grouped):
    get_sentiment(id, df)
    break

  0%|          | 0/150346 [02:05<?, ?it/s]


AttributeError: 'list' object has no attribute 'join'

In [21]:
vectorizer = CountVectorizer()

In [12]:
bag_of_words = vectorizer.fit_transform(df)

In [15]:
vectorizer.get_feature_names_out()

array(['10', 'advanced', 'ahead', 'all', 'almost', 'also', 'always',
       'amazing', 'and', 'anyone', 'as', 'assortment', 'at', 'attentive',
       'be', 'because', 'bikes', 'body', 'breakfast', 'buffet', 'butt',
       'can', 'casual', 'changed', 'check', 'chicken', 'choices',
       'clarion', 'class', 'classes', 'clean', 'clients', 'compares',
       'curry', 'cycle', 'cycling', 'day', 'delicious', 'desire', 'deter',
       'did', 'different', 'diner', 'do', 'don', 'dropping', 'easy',
       'eclectic', 'encouragement', 'even', 'every', 'evident',
       'expectations', 'face', 'family', 'favorite', 'fit', 'fitness',
       'for', 'fresh', 'fried', 'friendly', 'from', 'giving', 'glad',
       'go', 'good', 'grape', 'gyms', 'had', 'has', 'he', 'his', 'hotel',
       'ideas', 'in', 'instructors', 'is', 'it', 'jalapeño', 'kicking',
       'kinds', 'korma', 'lamb', 'large', 'leaves', 'leg', 'let', 'like',
       'line', 'll', 'long', 'lot', 'lots', 'make', 'makes', 'many',
       'mea

In [16]:
df_review_text.groupby('business_id')

KeyError: 'business_id'

In [3]:
def batch_iter(lst, n=2000, s=0):
    for i in range(s, len(lst), n):
        yield lst[i:i+n]

def batches(lst, n=2000, s=0):
    return list(batch_iter(lst, n, s))

In [7]:
texts = df_review_text['text'].tolist()

In [13]:
model_name = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name, truncation=True)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

roberta_base_remap = {
    'LABEL_2': 2,# 'Positive',
    'LABEL_1': 1,# 'Neutral',
    'LABEL_0': 0,# 'Negative'
}

sentiment = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    truncation=True,
    batch_size=32,
    padding="max_length",   # <—— makes everything exactly 512
    max_length=512,         # <—— hard limit
    device=0  # GPU = 0, CPU = -1
)
results = []

count = 0
for batch in tqdm(batches(texts, 2000, 212000)):

    out = sentiment(batch)
    sentiment_data = []

    for item in out:
        result = {
            'review_id': df_review_text.loc[count, 'review_id'],
            'sentiment': roberta_base_remap[item['label']],
            'score': item['score']
        }
        sentiment_data.append(result)
        count += 1
    results.append(sentiment_data)

    with open(EXPORT_PATH / f'sentiment_results_{count}.json', 'a') as f:
        json.dump(sentiment_data, f)
    break


urllib3.connectionpool: DEBUG: Resetting dropped connection: huggingface.co
urllib3.connectionpool: DEBUG: https://huggingface.co:443 "HEAD /cardiffnlp/twitter-roberta-base-sentiment/resolve/main/tokenizer_config.json HTTP/1.1" 404 0
urllib3.connectionpool: DEBUG: https://huggingface.co:443 "HEAD /cardiffnlp/twitter-roberta-base-sentiment/resolve/main/config.json HTTP/1.1" 307 0
urllib3.connectionpool: DEBUG: https://huggingface.co:443 "HEAD /api/resolve-cache/models/cardiffnlp/twitter-roberta-base-sentiment/daefdd1f6ae931839bce4d0f3db0a1a4265cd50f/config.json HTTP/1.1" 200 0
urllib3.connectionpool: DEBUG: https://huggingface.co:443 "HEAD /cardiffnlp/twitter-roberta-base-sentiment/resolve/main/tokenizer_config.json HTTP/1.1" 404 0
urllib3.connectionpool: DEBUG: https://huggingface.co:443 "GET /api/models/cardiffnlp/twitter-roberta-base-sentiment/tree/main/additional_chat_templates?recursive=False&expand=False HTTP/1.1" 404 64
urllib3.connectionpool: DEBUG: https://huggingface.co:443 "G

KeyboardInterrupt: 

In [ ]:
merged['close_date'] = merged.apply(
    lambda r: pd.NaT if r['is_open'] == 1 else r['close_date'],
    axis=1
)